# Phase 05 — XGBoost Classifier

**Phase 05 — CS114 Final Project: Vietnamese Elderly Scam Detection**  
**Tác giả:** Bùi Thị Mỷ Cẩm (MSSV: 25730013)  
**Ngày:** 2026-09-16  

---

## Mục tiêu

Huấn luyện XGBoost Classifier trên ma trận đặc trưng 5.015 chiều đã xuất sẵn từ Phase 03, xử lý mất cân bằng lớp bằng `scale_pos_weight ≈ 4.22`, tinh chỉnh hyperparameters bằng RandomizedSearchCV, đo 5 metrics, vẽ Confusion Matrix / ROC Curve / Learning Curve / Top-20 Feature Importance.

> **CRITICAL INVARIANT:** KHÔNG fit lại TF-IDF hay split lại dataset — dùng thẳng `X_train.pkl` / `X_test.pkl` từ Phase 03 để giữ consistency với Phase 04 và Phase 07.

## 0. Setup & Imports

In [ ]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import joblib
import xgboost as xgb

PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # kernel cwd = notebooks/ in some launchers
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import RANDOM_STATE, get_models_dir, get_reports_dir
from src.model_plots import (
    compute_metrics, plot_confusion_matrix, plot_learning_curve_chart,
    plot_roc_curve, plot_feature_importance,
)

DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR = get_reports_dir() / 'figures'
MODELS_DIR = get_models_dir()
print(f'RANDOM_STATE={RANDOM_STATE}, xgboost version={xgb.__version__}')

## 1. Nạp Feature Matrices Từ Phase 03 (KHÔNG split lại)

In [ ]:
X_train = joblib.load(DATA_DIR / 'X_train.pkl')   # (1754, 5015) sparse CSR
X_test  = joblib.load(DATA_DIR / 'X_test.pkl')    # (439, 5015)
y_train = joblib.load(DATA_DIR / 'y_train.pkl')
y_test  = joblib.load(DATA_DIR / 'y_test.pkl')

with open(DATA_DIR / 'feature_names.json', encoding='utf-8') as f:
    feature_names = json.load(f)

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'feature_names: {len(feature_names)} entries')

In [ ]:
from sklearn.model_selection import train_test_split

# Validation set (15% của X_train) — dùng cho early-stop / model selection, KHÔNG đụng X_test
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.15,
    random_state=RANDOM_STATE, stratify=y_train,
)
print(f'X_tr: {X_tr.shape}, X_val: {X_val.shape}')

## 2. Baseline XGBoost (scale_pos_weight bù mất cân bằng lớp)

In [ ]:
# scale_pos_weight = normal/scam trên tập train hiện tại
scale_pw = float((y_tr == 0).sum() / (y_tr == 1).sum())
print(f'scale_pos_weight = {scale_pw:.4f}')

xgb_baseline = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_pw, random_state=RANDOM_STATE,
    eval_metric='logloss', n_jobs=-1,
)
xgb_baseline.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

y_val_pred = xgb_baseline.predict(X_val)
y_val_proba = xgb_baseline.predict_proba(X_val)[:, 1]
baseline_metrics = compute_metrics(y_val, y_val_pred, y_val_proba)
print('--- XGBoost baseline (validation) ---')
for k, v in baseline_metrics.items():
    print(f'  {k:10s}: {v:.4f}')

## 3. Hyperparameter Tuning — RandomizedSearchCV (5-fold CV trên X_train)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

scale_pw_full = float((y_train == 0).sum() / (y_train == 1).sum())

xgb_param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
}

random_search = RandomizedSearchCV(
    xgb.XGBClassifier(
        scale_pos_weight=scale_pw_full, random_state=RANDOM_STATE,
        eval_metric='logloss', n_jobs=-1,
    ),
    param_distributions=xgb_param_dist,
    n_iter=25, cv=5, scoring='f1', n_jobs=-1,
    random_state=RANDOM_STATE, verbose=0,
)
t0 = time.time()
random_search.fit(X_train, y_train)
print(f'RandomizedSearchCV done in {time.time()-t0:.1f}s')
print(f'Best params: {random_search.best_params_}')
print(f'Best CV F1: {random_search.best_score_:.4f}')

## 4. Huấn luyện Model Cuối Cùng (best params, full X_train) & Đánh giá trên X_test

In [ ]:
xgb_model = random_search.best_estimator_  # đã tự refit trên toàn bộ X_train (refit=True mặc định)

y_test_pred = xgb_model.predict(X_test)
y_test_proba = xgb_model.predict_proba(X_test)[:, 1]
test_metrics = compute_metrics(y_test, y_test_pred, y_test_proba)

from sklearn.metrics import classification_report
print('--- XGBoost (tuned) — TEST SET ---')
for k, v in test_metrics.items():
    print(f'  {k:10s}: {v:.4f}')
print()
print(classification_report(y_test, y_test_pred, target_names=['Bình thường', 'Lừa đảo']))

## 5. Confusion Matrix & ROC Curve

In [ ]:
plot_confusion_matrix(
    y_test, y_test_pred, FIGURES_DIR, 'xgb_confusion_matrix.png',
    'Confusion Matrix — XGBoost (Test Set)',
)
plot_roc_curve(
    y_test, y_test_proba, FIGURES_DIR, 'xgb_roc_curve.png',
    'ROC Curve — XGBoost (Test Set)',
)

## 6. Learning Curve (Overfitting Check)

In [ ]:
plot_learning_curve_chart(
    xgb.XGBClassifier(**{**random_search.best_params_,
                         'scale_pos_weight': scale_pw_full,
                         'random_state': RANDOM_STATE,
                         'eval_metric': 'logloss', 'n_jobs': -1}),
    X_train, y_train, FIGURES_DIR, 'xgb_learning_curve.png',
    'Learning Curve — XGBoost (tuned)', cv=5, scoring='f1', random_state=RANDOM_STATE,
)

## 7. Feature Importance (Top-20)

In [ ]:
importances = xgb_model.feature_importances_
plot_feature_importance(
    importances, feature_names, FIGURES_DIR, 'xgb_feature_importance.png',
    'Top-20 Đặc trưng Quan trọng Nhất — XGBoost', top_n=20,
)

top_idx = np.argsort(importances)[::-1][:20]
for rank, i in enumerate(top_idx, 1):
    print(f'{rank:2d}. {feature_names[i]:30s} {importances[i]:.4f}')

## 8. Save Model & Metrics

In [ ]:
joblib.dump(xgb_model, MODELS_DIR / 'xgboost.pkl')

metrics_record = {
    'algorithm': 'XGBoost',
    'best_params': random_search.best_params_,
    'scale_pos_weight': scale_pw_full,
    'cv_f1_randomsearch': random_search.best_score_,
    'baseline_metrics_validation': baseline_metrics,
    'test_metrics': test_metrics,
    'random_state': RANDOM_STATE,
}
with open(MODELS_DIR / 'xgboost_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_record, f, ensure_ascii=False, indent=2)

print('Model saved: models/xgboost.pkl')
print('Metrics saved: models/xgboost_metrics.json')
print(json.dumps(metrics_record, ensure_ascii=False, indent=2))

## 9. Summary

**Phase 05 (XGBoost) — COMPLETED** ✅

Kết quả chi tiết xem output cell 'Save Model & Metrics' phía trên (`models/xgboost_metrics.json`). So sánh với baseline Naive Bayes (`models/naive_bayes_metrics.json`) ở Phase 04.

**Ready for:** Phase 06 (SVM — `05_model_svm.ipynb`), Phase 07 (So sánh tổng hợp — Thanh).